In [1]:
# Re-import after state reset
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.sample import sample_gen

In [2]:
# Паркет з ICESat-2 (вже з висотами)
icesat_path =  Path(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_lulc.parquet")
ice_gdf = gpd.read_parquet(icesat_path)
coords = [(geom.x, geom.y) for geom in ice_gdf.geometry]

In [3]:
# Папки з різними наборами растрів
attr_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/terrain_attributes")
geomorphon_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons")
hand_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/hand_outputs")
distance_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/distance_to_stream")
twi_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/flow_TWI")



In [4]:
# Формуємо списки шляхів
attr_paths = sorted(attr_folder.glob("*.tif"))
geomorphon_paths = sorted(geomorphon_folder.glob("*_geomorphons.tif"))
hand_paths = sorted(hand_folder.glob("*_hand_2000.tif"))
dist_paths = sorted(distance_folder.glob("*_distance_to_stream.tif"))
twi_paths = sorted(twi_folder.glob("*_twi.tif"))
# Словник назв форм

In [5]:


landform_names = {
    1: "Flat", 2: "Peak", 3: "Ridge", 4: "Shoulder", 5: "Spur",
    6: "Slope", 7: "Hollow", 8: "Footslope", 9: "Valley", 10: "Pit"
}

# 🔍 Атрибути, які нас цікавлять
important_attributes = {"slope", "curvature", "tpi", "tri", "roughness", "aspect"}

# 📦 Вибір файлів, які містять лише ці атрибути
filtered_attr_paths = [p for p in attr_paths if any(attr in p.stem for attr in important_attributes)]
# 🔁 Об'єднаний цикл по всіх растрах (атрибути, геоморфони, HAND, distance)
for path in filtered_attr_paths + geomorphon_paths + hand_paths + dist_paths + twi_paths:
    filename = path.stem

    if "geomorphons" in filename:
        # Geomorphons
        dem_name = "_".join(filename.split("_")[:2])
        col_class = f"{dem_name}_geomorphon"
        col_name = f"{dem_name}_landform"
        with rasterio.open(path) as src:
            sampled = list(src.sample(coords))
        geomorph_classes = [val[0] if val and val[0] > 0 else np.nan for val in sampled]
        geomorph_names = [
            landform_names.get(int(val), None) if not np.isnan(val) else None for val in geomorph_classes
        ]
        ice_gdf[col_class] = geomorph_classes
        ice_gdf[col_name] = geomorph_names
        print(f"🧭 Geomorphons: {col_class}, {col_name}")

    else:
        # Інші (атрибути, HAND, відстань)
        parts = filename.split("_")
        dem_name = "_".join(parts[:2])  # tan_dem
        attribute = parts[-1]           # slope / hand / tri / distance
        col_name = f"{dem_name}_{attribute}"

        with rasterio.open(path) as src:
            sampled = list(src.sample(coords))
            values = [val[0] if val and val[0] != src.nodata else np.nan for val in sampled]

        ice_gdf[col_name] = values
        print(f"📌 Додано: {col_name}")

# Зберігаємо об'єднаний результат
final_out = "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_features_for_model.parquet"
ice_gdf.to_parquet(final_out)
print(f"📦 Збережено до: {final_out}")


📌 Додано: alos_dem_aspect
📌 Додано: alos_dem_curvature
📌 Додано: alos_dem_roughness
📌 Додано: alos_dem_slope
📌 Додано: alos_dem_tpi
📌 Додано: alos_dem_tri
📌 Додано: aster_dem_aspect
📌 Додано: aster_dem_curvature
📌 Додано: aster_dem_roughness
📌 Додано: aster_dem_slope
📌 Додано: aster_dem_tpi
📌 Додано: aster_dem_tri
📌 Додано: copernicus_dеm_aspect
📌 Додано: copernicus_dеm_curvature
📌 Додано: copernicus_dеm_roughness
📌 Додано: copernicus_dеm_slope
📌 Додано: copernicus_dеm_tpi
📌 Додано: copernicus_dеm_tri
📌 Додано: fab_dem_aspect
📌 Додано: fab_dem_curvature
📌 Додано: fab_dem_roughness
📌 Додано: fab_dem_slope
📌 Додано: fab_dem_tpi
📌 Додано: fab_dem_tri
📌 Додано: nasa_dem_aspect
📌 Додано: nasa_dem_curvature
📌 Додано: nasa_dem_roughness


KeyboardInterrupt: 

In [1]:
from pathlib import Path

# 📂 Папка з усіма атрибутами
attr_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/terrain_attributes")
attr_paths = sorted(attr_folder.glob("*.tif"))

# 🔍 Атрибути, які нас цікавлять
important_attributes = {"slope", "curvature", "tpi", "tri", "roughness", "aspect"}

# 📦 Вибір файлів, які містять лише ці атрибути
filtered_attr_paths = [p for p in attr_paths if any(attr in p.stem for attr in important_attributes)]

# 📋 Перевірка
for path in filtered_attr_paths:
    print(path.name)

alos_dem_utm32635_aspect.tif
alos_dem_utm32635_curvature.tif
alos_dem_utm32635_roughness.tif
alos_dem_utm32635_slope.tif
alos_dem_utm32635_tpi.tif
alos_dem_utm32635_tri.tif
aster_dem_utm32635_aspect.tif
aster_dem_utm32635_curvature.tif
aster_dem_utm32635_roughness.tif
aster_dem_utm32635_slope.tif
aster_dem_utm32635_tpi.tif
aster_dem_utm32635_tri.tif
copernicus_dеm_utm32635_aspect.tif
copernicus_dеm_utm32635_curvature.tif
copernicus_dеm_utm32635_roughness.tif
copernicus_dеm_utm32635_slope.tif
copernicus_dеm_utm32635_tpi.tif
copernicus_dеm_utm32635_tri.tif
fab_dem_utm32635_aspect.tif
fab_dem_utm32635_curvature.tif
fab_dem_utm32635_roughness.tif
fab_dem_utm32635_slope.tif
fab_dem_utm32635_tpi.tif
fab_dem_utm32635_tri.tif
nasa_dem_utm32635_aspect.tif
nasa_dem_utm32635_curvature.tif
nasa_dem_utm32635_roughness.tif
nasa_dem_utm32635_slope.tif
nasa_dem_utm32635_tpi.tif
nasa_dem_utm32635_tri.tif
srtm_dem_utm32635_aspect.tif
srtm_dem_utm32635_curvature.tif
srtm_dem_utm32635_roughness.tif
srtm_d